# 04 · Persistence in LangGraph - threads, checkpoints & time travel

> Workshop module · builds on last class (`agent-builder: thinking in langgraph`).
> Today we make our graphs **remember**.

**The big idea:** a plain LangGraph run is *stateless* - when `invoke()` returns, the state is gone.
Attach a **checkpointer** and LangGraph saves a snapshot of the state at **every step**, keyed by a
**`thread_id`**. That single change unlocks:

- 🧠 **Memory** - multi-turn conversations that remember earlier turns
- 👤 **Human-in-the-loop** - pause, inspect, approve, resume
- ⏪ **Time travel** - replay or fork from any past step
- 🛟 **Fault tolerance** - resume from the last good step after a crash

We'll go slowly with lots of tiny, runnable demos.

![checkpoints](images/checkpoints.jpg)

> ⏸️ **Scope for today:** short-term memory via the **checkpointer + thread**.
> The *long-term* **Memory Store** (cross-thread memory) is the *next* session - we stop right before it.

## 0 · Setup

We load the OpenAI key from your local `.env` (already in this folder).

In [14]:
import os
# --- Load API keys from the project .env, no matter how/where Jupyter was started ---
_ENV_PATH = "/Users/datasense/Desktop/langgrapgh-agent/.env"
for _line in open(_ENV_PATH):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        os.environ[_k.strip()] = _v.strip().strip('"').strip("'")
print("Loaded keys:", [k for k in ("OPENAI_API_KEY", "TAVILY_API_KEY", "SERPAPI_API_KEY") if os.getenv(k)])
  # reads .env in this folder
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
print("OpenAI key loaded")

Loaded keys: ['OPENAI_API_KEY', 'TAVILY_API_KEY', 'SERPAPI_API_KEY']
OpenAI key loaded


In [ ]:
from IPython.display import display, HTML
import uuid

def show_mermaid(diagram: str, height: int = 400):
    """Render a Mermaid diagram inline. Requires internet for CDN."""
    uid = str(uuid.uuid4())[:8]
    display(HTML(f"""
<div id="mg-{uid}" style="background:#fff;padding:12px;border-radius:8px;
     border:1px solid #e0e0e0;max-width:860px;overflow-x:auto;">
  <pre class="mermaid" style="margin:0;">{diagram}</pre>
</div>
<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
  mermaid.initialize({{ startOnLoad: false, theme: 'default',
    sequence: {{ mirrorActors: false }},
    flowchart: {{ curve: 'basis' }} }});
  await mermaid.run({{ querySelector: '#mg-{uid} .mermaid' }});
</script>
"""))

print("show_mermaid() ready")

## 1 · A chatbot with no memory

Our running example all day: a simple **chatbot** powered by `gpt-4o-mini`.  
One node, one job — call the LLM, return the reply.

**The core question:** does the bot remember who you are between two calls?

```
Without a checkpointer:

  invoke("Hi! I'm Satvik")      -> [START]->[chat]->[END]   STATE WIPED ❌
  invoke("What's my name?")     -> [START]->[chat]->[END]   STATE WIPED ❌
                                   starts from a blank slate every time
```

Let's prove it with a real LLM call.

In [15]:
from IPython.display import Image, display

def show(graph):
    "Render a LangGraph as an inline mermaid PNG (falls back to ASCII if offline)."
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(graph.get_graph().draw_ascii())

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, START, END

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

def chat_node(state: MessagesState):
    reply = llm.invoke(state["messages"])
    return {"messages": [reply]}

chat_builder = StateGraph(MessagesState)
chat_builder.add_node("chat", chat_node)
chat_builder.add_edge(START, "chat")
chat_builder.add_edge("chat", END)

bot_no_memory = chat_builder.compile()   # no checkpointer -> no memory
show(bot_no_memory)
print("\nMessagesState uses add_messages reducer -> messages accumulate (but only within one invoke())")

### What happens inside each `invoke()` with no checkpointer

In [ ]:
show_mermaid("""
sequenceDiagram
    actor You
    participant G as Graph (no checkpointer)

    rect rgb(255,235,235)
        Note over You,G: invoke() #1
        You->>G: Hi! I am Satvik
        G->>G: START → chat → END
        G-->>You: Hi Satvik! How can I help?
        Note right of G: ❌ state discarded — nothing saved
    end

    rect rgb(255,235,235)
        Note over You,G: invoke() #2 — blank slate
        You->>G: What is my name?
        G->>G: START → chat → END
        G-->>You: I don't know your name
        Note right of G: ❌ state discarded — nothing saved
    end
""")

In [ ]:
# Turn 1 - introduce yourself
r1 = bot_no_memory.invoke({
    "messages": [{"role": "user", "content": "Hi! My name is Satvik and I love LangGraph."}]
})
print("Turn 1 ->", r1["messages"][-1].content[:80])

# Turn 2 - ask bot to recall. It gets a FRESH empty messages list - has no idea.
r2 = bot_no_memory.invoke({
    "messages": [{"role": "user", "content": "What's my name and what do I love?"}]
})
print("Turn 2 ->", r2["messages"][-1].content[:80])

print("\n❌ The bot forgot. Each invoke() wipes the state the instant it returns.")
print("   (We can't even call get_state() - there's no cabinet to look in.)")

try:
    bot_no_memory.get_state({"configurable": {"thread_id": "test"}})
except Exception as e:
    print(f"\n   get_state() raises: {type(e).__name__}: {e}")

## 2 · Add a checkpointer + a thread

One line change at compile time: `checkpointer=InMemorySaver()`.  
One addition to the config: a `thread_id`.

That's it. Now every step writes a snapshot to the checkpointer.  
Think of the checkpointer as a **filing cabinet** and the `thread_id` as the **drawer label**.  
Every `invoke()` opens that drawer, loads history, runs, then saves the new snapshot back.

In [ ]:
show_mermaid("""
sequenceDiagram
    actor You
    participant G as Graph
    participant S as InMemorySaver<br/>thread "alice"

    rect rgb(220,255,220)
        Note over You,S: invoke() #1
        You->>G: Hi! I am Satvik
        G->>G: START → chat → END
        G->>S: ✅ SAVE checkpoint_1<br/>[HumanMsg, AIMsg]
        G-->>You: Hi Satvik!
    end

    rect rgb(220,255,220)
        Note over You,S: invoke() #2 — loads history first
        You->>G: What is my name?
        G->>S: load latest checkpoint
        S-->>G: [HumanMsg, AIMsg]
        G->>G: chat sees full history
        G->>S: ✅ SAVE checkpoint_2<br/>[HumanMsg, AIMsg, HumanMsg, AIMsg]
        G-->>You: Your name is Satvik!
    end
""")

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
bot = chat_builder.compile(checkpointer=checkpointer)   # SAME graph, now WITH memory

alice = {"configurable": {"thread_id": "alice"}}

# Turn 1 - introduce yourself
r1 = bot.invoke({
    "messages": [{"role": "user", "content": "Hi! My name is Satvik and I love LangGraph."}]
}, alice)
print("Turn 1 ->", r1["messages"][-1].content[:80])

# Turn 2 - ask to recall. Checkpointer loads the full history before calling the LLM.
r2 = bot.invoke({
    "messages": [{"role": "user", "content": "What's my name and what do I love?"}]
}, alice)
print("Turn 2 ->", r2["messages"][-1].content[:80])

print("\n✅ The bot remembered! The checkpointer loaded the history before each call.")

## 3 · Read the saved state: `get_state`

`get_state(config)` returns a **`StateSnapshot`** - the latest checkpoint for this thread.

Key fields:

| field | meaning |
|---|---|
| `values` | the actual state at this checkpoint |
| `next` | which node(s) run next; **empty `()` = finished** |
| `config` | identifies the checkpoint (`thread_id`, `checkpoint_id`) |
| `metadata` | `step` counter, `source`, and the `writes` that produced it |
| `created_at` | timestamp |
| `parent_config` | the previous checkpoint (the chain backwards) |

In [ ]:
snap = bot.get_state(alice)

msgs = snap.values["messages"]
print("values   :", f"messages list with {len(msgs)} items")
print("last msg :", f'[{msgs[-1].type}] "{msgs[-1].content[:55]}"')
print("next     :", snap.next, "  <- empty tuple means the run is complete")
print("step     :", snap.metadata["step"])
print("checkpoint:", snap.config["configurable"]["checkpoint_id"])

## 4 · The whole story: `get_state_history`

Every **super-step** (one "tick" where scheduled nodes run) writes a checkpoint.
`get_state_history` returns them **newest → oldest**, so you can see the graph's entire timeline.

![get_state](images/get_state.jpg)

In [ ]:
history = list(bot.get_state_history(alice))
print(f"{len(history)} checkpoints saved for thread 'alice':\n")
for snap in history:
    msgs = snap.values.get("messages", [])
    last = f'"{msgs[-1].content[:40]}"' if msgs else "-"
    print(f"  step {snap.metadata['step']:>2}  next={str(snap.next):<14}  msgs={len(msgs)}  last={last}")

**Read it bottom-up:** step -1 is the raw input, then each subsequent step ran one node.
The final step has `next=()` (finished). Every row is a point we can jump back to - that's time travel.

Notice the message count growing: each checkpoint is a cumulative snapshot of the full state, not just the diff.

## 5 · Why the thread matters - accumulation vs isolation

Two experiments that make "thread = one conversation" click.

**5a. Same thread again** - state is still in the checkpointer, so the next message appends to the existing history.

In [ ]:
# Continue alice's conversation with a 3rd message on the SAME thread
r3 = bot.invoke({
    "messages": [{"role": "user", "content": "Given what you know about me, what programming stack do I probably use?"}]
}, alice)
print("Turn 3 ->", r3["messages"][-1].content[:100])

n = len(bot.get_state(alice).values["messages"])
print(f"\n-> alice's thread now has {n} messages - the conversation keeps accumulating")

In [ ]:
# A DIFFERENT thread starts completely fresh - knows nothing about Satvik
bob = {"configurable": {"thread_id": "bob"}}
r = bot.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, bob)
print("Bob's bot ->", r["messages"][-1].content[:80])

n = len(bot.get_state(bob).values["messages"])
print(f"\n-> bob's thread has {n} message(s) - blank slate, no memory of alice's thread")

### Two threads, two independent timelines — one shared cabinet

Same `bot`, same `chat_node`, same `InMemorySaver`.  
Different `thread_id` = completely isolated conversation.  
Switching `thread_id` is how you start a new conversation without losing old ones.

In [ ]:
show_mermaid("""
flowchart TD
    C["🗄️ InMemorySaver<br/>one shared cabinet in RAM"]

    C --> A["📂 thread = 'alice'<br/>ckpt_1 → ckpt_2 → ckpt_3"]
    C --> B["📂 thread = 'bob'<br/>ckpt_1"]

    A --> A1["6 messages · keeps growing<br/>HumanMsg · AIMsg · HumanMsg · AIMsg..."]
    B --> B1["2 messages · fresh start<br/>knows nothing about alice"]

    style C fill:#2c3e50,color:#fff
    style A fill:#27ae60,color:#fff
    style B fill:#e67e22,color:#fff
    style A1 fill:#d5f5e3,color:#2c3e50
    style B1 fill:#fdebd0,color:#2c3e50
""")

> 🔑 **Takeaway:** the checkpointer is shared, but each `thread_id` has its own independent timeline.
> Same thread = continue; new thread = blank slate.

## 6 · Peek inside: what the checkpointer actually stored

`get_state` opened the drawer and showed us the *latest* snapshot.  
Let's look at the **full message list** - every message across all of alice's turns, exactly as the LLM sees it on each call.

In [ ]:
msgs = bot.get_state(alice).values["messages"]
print(f"alice's thread - {len(msgs)} messages in the checkpointer:\n")
for i, m in enumerate(msgs, 1):
    icon = "YOU:" if m.type == "human" else "BOT:"
    print(f"  [{i}] {icon:<5}  {m.content[:75]}")

## 6.5 · ⚠️ "My bot forgot me!" — when memory *looks* gone but isn't

This is the **#1 thing that confuses people** about persistence. The bot works, then suddenly acts like it has amnesia. **9 times out of 10 the memory was never lost** — you either looked in the *wrong place* or never *turned saving on*.

`get_state(config)` opens exactly **one drawer**: the one whose `thread_id` you passed in.  
Wrong id → wrong (or empty) drawer → "it forgot me!"

Let's interrogate the **3 usual suspects**. 🕵️

In [ ]:
show_mermaid("""
flowchart LR
    Q["get_state<br/>thread_id = 'user-satvk'"]

    subgraph cabinet["🗄️ InMemorySaver — one shared cabinet"]
        D1["📂 user-satvik<br/>✅ BANANA-42 is here<br/>(untouched)"]
        D2["📂 user-satvk  ← TYPO<br/>❌ empty drawer<br/>(silently created)"]
        D3["📂 alice<br/>✅ messages here"]
    end

    Q -->|"opens THIS drawer"| D2

    style D1 fill:#27ae60,color:#fff
    style D2 fill:#e74c3c,color:#fff
    style D3 fill:#27ae60,color:#fff
    style Q fill:#e67e22,color:#000
""")

In [ ]:
# ── Suspect #1: a typo in the thread_id ──────────────────
# Satvik tells the bot a secret, on HIS thread:
satvik = {"configurable": {"thread_id": "user-satvik"}}
bot.invoke({"messages": [{"role": "user", "content": "Remember this: my secret code is BANANA-42."}]}, satvik)

# Later, another request arrives - but with a TYPO in the thread id (missing the 'i'):
typo = {"configurable": {"thread_id": "user-satvk"}}
r = bot.invoke({"messages": [{"role": "user", "content": "What's my secret code?"}]}, typo)

print("Asked on thread 'user-satvk' (the typo):")
print("  ->", r["messages"][-1].content)
print()
print("😱 Looks like the bot forgot! ... but did it really?")

**No — the memory is perfectly safe.** You just knocked on the wrong drawer. Don't take my word for it — let's open *both* drawers ourselves:

In [ ]:
print("📂 drawer 'user-satvik' (the real one) holds:")
for m in bot.get_state(satvik).values["messages"]:
    print(f"     {m.type:>6}: {m.content[:55]}")

n = len(bot.get_state(typo).values["messages"])
print(f"\n📂 drawer 'user-satvk' (the typo) holds: {n} message(s)  <- empty/new, that's why it 'forgot'")
print("\n✅ Nothing was lost. The memory sat safely in the other drawer the whole time.")

### Suspect #2: you forgot the checkpointer at compile time

A `thread_id` **does nothing on its own**. If you compile the graph **without** a checkpointer, there is no cabinet at all - every run starts from zero and there is nothing to read back. This is the *opposite* trap: people add a `thread_id`, forget `checkpointer=...`, and wonder why nothing sticks.

In [ ]:
# Same chatbot graph, but compiled WITHOUT a checkpointer:
forgetful = chat_builder.compile()          # <-- notice: no checkpointer=...

cfg = {"configurable": {"thread_id": "anything"}}
forgetful.invoke({"messages": [{"role": "user", "content": "Hi, my name is Satvik."}]}, cfg)
r = forgetful.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, cfg)
print("no-checkpointer bot ->", r["messages"][-1].content)

# There isn't even anything to read back:
try:
    forgetful.get_state(cfg)
except Exception as e:
    print("\nget_state(...) raises:", type(e).__name__, "-", e)

print("\n👉 Here the memory is REALLY absent - because saving was never turned on.")

### Suspect #3: a brand-new `InMemorySaver` (the "I re-ran my cell" trap)

`InMemorySaver` lives in RAM, and **every `InMemorySaver()` is its own separate cabinet**. Re-running your "compile" cell builds a *new, empty* cabinet - so the same `thread_id` now points at a fresh, empty drawer. (And restarting the kernel wipes it entirely - that's exactly why Section 9 switches to `SqliteSaver`.)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# First cabinet:
bot_v1 = chat_builder.compile(checkpointer=InMemorySaver())
bot_v1.invoke({"messages": [{"role": "user", "content": "My name is Satvik."}]},
              {"configurable": {"thread_id": "t"}})

# Oops - we "re-ran the compile cell", making a SECOND, empty cabinet (same code, new saver):
bot_v2 = chat_builder.compile(checkpointer=InMemorySaver())
r = bot_v2.invoke({"messages": [{"role": "user", "content": "What's my name?"}]},
                  {"configurable": {"thread_id": "t"}})
print("after re-creating the saver ->", r["messages"][-1].content)

print("\n👉 Same thread_id, but a different cabinet = looks forgotten.")
print("   Reuse ONE saver instance to keep the memory.")

### 🧯 Before you ever say "persistence is broken", check these things

| Symptom | Real cause | Fix |
|---|---|---|
| Bot forgot mid-conversation | `thread_id` changed (typo / new value) | Pass the **same** `thread_id` every turn |
| Nothing persists at all | Compiled **without** a checkpointer | `compile(checkpointer=...)` |
| `get_state` raises an error | No checkpointer on this graph | Same as above |
| Forgets after re-running a cell | A **new** `InMemorySaver` was created | Reuse one saver, or use `SqliteSaver` |
| Forgets after a kernel restart | `InMemorySaver` is RAM-only | Use `SqliteSaver` / `PostgresSaver` (Section 9) |

> 🧠 **Mantra:** *"It didn't forget - I opened the wrong drawer, or I never bought the cabinet."*
>
> Persistence needs **both**: a **checkpointer** (the cabinet) **and** a stable **`thread_id`** (the drawer). Miss either one and the memory looks gone.

## 7 · Time travel - replay from a past checkpoint

Because every step is saved, we can **resume the graph from any past checkpoint**.

> **Note:** For this section we use a simple state machine (not the chatbot) so the steps are easy to read.
> The concepts transfer directly - `thread_id`, `get_state_history`, and `invoke(None, checkpoint_cfg)` work identically on any graph.

![replay](images/re_play.png)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

class OrderState(TypedDict):
    status: str
    log: Annotated[list[str], add]

def bake(state: OrderState):
    return {"status": "baked", "log": ["baked the pizza"]}

def deliver(state: OrderState):
    return {"status": "delivered", "log": ["delivered to the door"]}

order_builder = StateGraph(OrderState)
order_builder.add_node(bake)
order_builder.add_node(deliver)
order_builder.add_edge(START, "bake")
order_builder.add_edge("bake", "deliver")
order_builder.add_edge("deliver", END)

graph = order_builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"status": "ordered", "log": []}, config)
print("pizza order graph ready")
print("state:", graph.get_state(config).values)

In [ ]:
# use our memory graph (thread "1"). Find the checkpoint taken right BEFORE deliver ran.
hist = list(graph.get_state_history({"configurable": {"thread_id": "1"}}))
before_deliver = next(s for s in hist if s.next == ("deliver",))
print("resuming from step", before_deliver.metadata["step"], "where next =", before_deliver.next)

# invoke with input=None and that checkpoint's config -> replays 'deliver' forward
resumed = graph.invoke(None, before_deliver.config)
print("replayed result:", resumed)

## 8 · Forking - `update_state`

Time travel becomes powerful when you **change** the past. `update_state` writes a *new* checkpoint
on top of a chosen one - a branch in the timeline. Great for "what if the state had been different?"
and for human edits / approvals.

In [ ]:
fork_cfg = {"configurable": {"thread_id": "order-99"}}
graph.invoke({"status": "ordered", "log": []}, fork_cfg)
print("before update:", graph.get_state(fork_cfg).values)

# a human steps in and CANCELS the order -> writes a NEW checkpoint (metadata source='update')
graph.update_state(fork_cfg, {"status": "CANCELLED", "log": ["order cancelled by support"]})
print("after  update:", graph.get_state(fork_cfg).values)
print("source of newest checkpoint:", graph.get_state(fork_cfg).metadata["source"])

## 9 · Durable memory across restarts - `SqliteSaver`

`InMemorySaver` vanishes when the kernel dies. For anything real you want a checkpointer backed by a
database. `SqliteSaver` writes to a file, so state **survives a process restart**.
(In production you'd swap in `PostgresSaver` - same interface.)

![checkpoints full story](images/checkpoints_full_story.jpg)

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("workshop_checkpoints.sqlite", check_same_thread=False)
durable_graph = order_builder.compile(checkpointer=SqliteSaver(conn))
durable_graph.invoke({"status": "ordered", "log": []}, {"configurable": {"thread_id": "persist-me"}})
print("wrote state to workshop_checkpoints.sqlite")

In [ ]:
conn2 = sqlite3.connect("workshop_checkpoints.sqlite", check_same_thread=False)
reloaded = order_builder.compile(checkpointer=SqliteSaver(conn2))

snap = reloaded.get_state({"configurable": {"thread_id": "persist-me"}})
print("recovered from disk after 'restart':", snap.values)

## Recap

| concept | what it gives you |
|---|---|
| **`thread_id`** | the key for one session's timeline |
| **checkpointer** (`InMemorySaver`, `SqliteSaver`, `PostgresSaver`) | saves a snapshot every super-step |
| **`get_state`** | the latest `StateSnapshot` (`values`, `next`, `metadata`...) |
| **`get_state_history`** | the full newest→oldest timeline |
| **replay** (`invoke(None, checkpoint_cfg)`) | resume from any past step |
| **`update_state`** | edit/fork the timeline (human-in-the-loop) |

**Mental model:** *checkpointer = the disk, `thread_id` = the file name, checkpoint = a saved version.*

### ⏭️ Next session
**Memory Store** - long-term memory that is **shared across threads** (e.g. remember a user's food
preference in *every* conversation, not just one). `store.put(...)` / `store.search(...)`, namespaces,
and semantic search. That's where we pick up next time.

### 🏋️ Exercises
1. Add a third thread and confirm isolation.
2. In the chatbot, find the `checkpoint_id` after turn 1 and replay turn 2 from it.
3. Use `update_state` on the chatbot to inject a fake assistant message, then continue the conversation.
4. **Break it on purpose (the drawer game):** chat on thread `"x"`, then ask the same question on thread `"X"` (capital). Explain what you see using the filing-cabinet model from Section 6.5, then prove the first thread's memory is still intact with `get_state`.
